# Experiment 5.4.2 — Phase-conditioned readout mechanism search

Analysis-only notebook. Training and Slurm execution live in the experiment script. Mechanism/capacity selection is validation-only; final test is opened only after `selection.json` is locked.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'AGENTS.md').exists():
            return candidate
    raise FileNotFoundError('repository root not found')

ROOT = find_repo_root() / 'notebooks' / 'artifacts' / 'experiment_5_4_2_phase_conditioned_readout' / 'phase_conditioned_readout_v1'
ROOT

In [ ]:
def csv(name):
    path = ROOT / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

screen_runs = csv('screen_runs.csv')
screen_paired = csv('screen_paired_deltas.csv')
refine_runs = csv('refine_runs.csv')
final_runs = csv('final_runs.csv')
ablation_runs = csv('ablation_runs.csv')
gate_activity = csv('gate_activity.csv')
residual_metrics = csv('residual_metrics.csv')
rescue_harm = csv('rescue_harm.csv')
prefix_ba = csv('prefix_ba.csv')
selection = json.loads((ROOT / 'selection.json').read_text()) if (ROOT / 'selection.json').exists() else {}
manifest = json.loads((ROOT / 'manifest.json').read_text()) if (ROOT / 'manifest.json').exists() else {}
{'screen': len(screen_runs), 'refine': len(refine_runs), 'final': len(final_runs), 'selection': selection}

## Stage A validation screen

In [ ]:
if not screen_runs.empty:
    summary = screen_runs.groupby('condition', as_index=False).agg(mean_val_ba=('val_balanced_accuracy','mean'), mean_delta=('delta_val_balanced_accuracy_vs_what','mean'))
    display(summary.sort_values('mean_delta', ascending=False))
    fig, ax = plt.subplots(figsize=(10,4))
    order = summary.sort_values('mean_delta', ascending=False)
    ax.bar(order['condition'], order['mean_delta'])
    ax.axhline(0, linewidth=1)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Paired validation BA delta vs WHAT')
    fig.tight_layout()

## Locked final test and temporal ablations

In [ ]:
if final_runs.empty:
    print('Final test artifacts are not available yet.')
else:
    display(final_runs)
    print('Mean paired test BA delta:', final_runs['delta_test_ba_vs_base'].mean())

if not ablation_runs.empty:
    display(ablation_runs.groupby('ablation')['test_balanced_accuracy'].agg(['mean','sem']))

## Residual, rescue/harm, prefix BA, and bank-gate diagnostics

In [ ]:
if not residual_metrics.empty: display(residual_metrics)
if not rescue_harm.empty: display(rescue_harm)
if not prefix_ba.empty:
    summary = prefix_ba.groupby(['prefix_fraction','model'], as_index=False)['balanced_accuracy'].mean()
    fig, ax = plt.subplots(figsize=(8,4))
    for model, group in summary.groupby('model'):
        ax.plot(group['prefix_fraction'], group['balanced_accuracy'], marker='o', label=model)
    ax.legend(); ax.set_xlabel('Valid gesture progress'); ax.set_ylabel('BA')

phase = gate_activity[gate_activity['metric'] == 'centered_gate_by_relative_phase'] if not gate_activity.empty else pd.DataFrame()
if not phase.empty:
    mean_phase = phase.groupby(['phase_bin','bank'], as_index=False)['value'].mean()
    fig, ax = plt.subplots(figsize=(8,4))
    for bank, group in mean_phase.groupby('bank'):
        ax.plot(group['phase_bin'], group['value'], marker='o', label=f'bank {int(bank)}')
    ax.axhline(0, linewidth=1); ax.legend(); ax.set_xlabel('Relative progress bin'); ax.set_ylabel('Centered gate')